In [ ]:
import json, re, subprocess, sys, zipfile, io
from collections import defaultdict
from pathlib import Path
import pandas as pd

def clone_repo(github_url, dest="cloned_repo"):
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"{dest_path} already exists, skipping clone.")
        return dest_path
    result = subprocess.run(["git", "clone", "--depth", "1", github_url, str(dest_path)],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print("CLONE FAILED:", result.stderr); sys.exit(1)
    print(f"Cloned {github_url} -> {dest_path}")
    return dest_path

repo_dir = clone_repo("https://github.com/your-org/your-service.git")

In [ ]:
def build_class_index_from_jar_folder(folder_path):
    """class -> ARTIFACT only (no group). Ground truth from real .class
    entries, no cross-referencing against declared dependencies, so
    transitively-used libraries are detected correctly too."""
    folder = Path(folder_path)
    jar_files = list(folder.glob("*.jar"))
    print(f"Found {len(jar_files)} jar file(s) in {folder}")
    class_index = {}
    for jar_path in jar_files:
        m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_path.name)
        artifact = m.group(1) if m else jar_path.stem
        with zipfile.ZipFile(jar_path) as jar:
            for name in jar.namelist():
                if name.endswith(".class") and "/" in name and "module-info" not in name:
                    class_index[name[:-6].replace("/", ".")] = artifact
    return class_index

class_index = build_class_index_from_jar_folder("/path/to/your/local/jar/folder")

def build_class_index_from_boot_jar(boot_jar_path):
    class_index = {}
    with zipfile.ZipFile(boot_jar_path) as outer:
        lib_entries = [n for n in outer.namelist() if n.startswith("BOOT-INF/lib/") and n.endswith(".jar")]
        for entry_name in lib_entries:
            jar_filename = entry_name.split("/")[-1]
            inner_bytes = outer.read(entry_name)
            m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_filename)
            artifact = m.group(1) if m else jar_filename.replace(".jar", "")
            with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
                for name in inner.namelist():
                    if name.endswith(".class") and "/" in name and "module-info" not in name:
                        class_index[name[:-6].replace("/", ".")] = artifact
    return class_index

# class_index = build_class_index_from_boot_jar("/path/to/your-service.jar")

In [ ]:
def load_category_map(path):
    df = pd.read_excel(path, sheet_name="library_categories")
    return dict(zip(df["artifact"], df["category"]))

CATEGORY = load_category_map("playbook.xlsx")  # sheet columns: artifact | category

In [ ]:
IMPORT_RE = re.compile(r"^\s*import\s+(?:static\s+)?([\w.]+)\s*;", re.MULTILINE)

def scan_source(repo_dir, class_index):
    files = [f for f in list(Path(repo_dir).rglob("*.java")) + list(Path(repo_dir).rglob("*.groovy"))
             + list(Path(repo_dir).rglob("*.kt")) if "/build/" not in str(f)]
    print(f"Scanning {len(files)} source file(s)")
    usage_count = defaultdict(int)
    for f in files:
        for imp in IMPORT_RE.findall(f.read_text()):
            artifact = class_index.get(imp)
            if artifact:
                usage_count[artifact] += 1
    return usage_count

usage_count = scan_source(repo_dir, class_index)
print(dict(usage_count))

In [ ]:
clusters = defaultdict(list)
for artifact in usage_count:
    clusters[CATEGORY.get(artifact, "uncategorized")].append(artifact)
duplicate_clusters = {c: v for c, v in clusters.items() if len(v) > 1}

print("DUPLICATE-FUNCTIONALITY CLUSTERS (for human review):")
for cat, artifacts in duplicate_clusters.items():
    print(f"  [{cat}] {artifacts}")